In [1]:
# !pip install --upgrade google-play-scraper
# !pip install pyqt5
# !pip install --upgrade beautifulsoup4
# !pip install pyqtwebengine
# !pip install pandas
# !pip install selenium

In [2]:
# from google_play_scraper import collection
# from google_play_scraper.constants import Collection, Category

import time
import json
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google_play_scraper import app, permissions



#Identified risk categories in each app's permissions
risk_categories = {
    "overlay": ["draw over other apps"],
    "background": ["run at startup", "prevent device from sleeping"],
    "bluetooth": ["pair with Bluetooth devices", "access Bluetooth settings"]
}


In [3]:
def collect_app_data():
    error_apps = {}

    with open("family_apps_strict_enriched.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    flag_dict = {}
    seen_apps = set()

    for _, row in df.iterrows():
        app_id = row.get("appId")

        if not app_id or app_id in seen_apps:
            continue
        seen_apps.add(app_id)

        try:
            perm = permissions(app_id)
            url = row.get("privacyPolicy")
            title = row.get("title")
            genre = row.get("genre")

            if not url:
                continue

            response = requests.get(url, timeout=(3, 5))
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")

            for tag in soup(["script", "style", "noscript"]):
                tag.decompose()

            policy_text = soup.get_text("\n").lower()
            other_permissions = perm.get("Other", [])

            flag_dict[app_id] = {
                "title": title,
                "genre": genre,
                "permissions": other_permissions,
                "policy_text": policy_text,
            }

        except requests.RequestException as e:
            print(f"Request error for {app_id}: {e}")
            error_apps[app_id] = str(e)
        except Exception as e:
            print(f"Error fetching {app_id}: {e}")
            error_apps[app_id] = str(e)

    if error_apps:
        print(f"Skipped {len(error_apps)} apps due to fetch/parse errors.")

    return flag_dict

In [4]:
import json
import os

CACHE_FILE = "app_data_cache.json"
def load_flag_dict():
    if os.path.exists(CACHE_FILE):
        print("Loading from cache...")
        with open(CACHE_FILE, "r") as f:
            return json.load(f)
    
    print("Fetching app data...")
    flag_dict = collect_app_data()
    
    with open(CACHE_FILE, "w") as f:
        json.dump(flag_dict, f, indent=2)
    
    return flag_dict

In [5]:
# Track apps that request at least one Bluetooth-related permission
bluetooth_apps = []

# Call data collection function
flag_dict = load_flag_dict()

bluetooth_risk_perms = set(risk_categories.get("bluetooth", []))

# For each app, check permission overlap with risk categories
for app_id, data in flag_dict.items():
    perms = set(data.get("permissions", []))
    title = data.get("title") or app_id

    if perms & bluetooth_risk_perms:
        bluetooth_apps.append(title)

total_apps = len(flag_dict)
bluetooth_rate = len(bluetooth_apps) / total_apps if total_apps else 0.0

print(f"Apps that ask for Bluetooth permission: {len(bluetooth_apps)} / {total_apps} = {bluetooth_rate:.4f}")

print("\nBluetooth Permission Rate:", round(bluetooth_rate * 100, 2), "%")


Loading from cache...
Apps that ask for Bluetooth permission: 40 / 585 = 0.0684

Bluetooth Permission Rate: 6.84 %


## Cell Below may be adapted or deleted in the future. I'll keep it here for now. Right now it just checks for the keyword "bluetooth", but the cell below this one identifies key bluetooth phrases not just the word bluetooth.

In [6]:

#list of undisclosed apps (Bluetooth)
bluetooth_undisclosed = []

#create perms and policy variables
for app_id, data in flag_dict.items():
    perms = data["permissions"]
    policy = data["policy_text"]


    #Boolean check for bluetooth risk and if it mentions Bluetooth in the policy.
    #If not, add it to the undisclosed apps list
    has_bt = any(p in risk_categories["bluetooth"] for p in perms)
    mentions_bt = "bluetooth" in policy

    if has_bt and not mentions_bt:
        bluetooth_undisclosed.append(data["title"])


print("Bluetooth Non-Disclosure Cases:")
print(f"{len(bluetooth_undisclosed)} / {len(flag_dict)}")


Bluetooth Non-Disclosure Cases:
40 / 585


In [7]:
#NLP and BERT integration
#Bluetooth Disclosure = policy mentions scanning, connecting, pairing,
#or communicating with nearby devices wirelessly, regardless of the keyword "Bluetooth" being mentioned

BLUETOOTH_TERMS = [
  "bluetooth", "ble", "bluetooth low energy",
  "nearby devices", "device discovery", "scan for devices",
  "scanning for devices", "pairing", "paired",
  "ibeacon", "eddystone", "beacon"
]

bt_disclosed = []

bluetooth_undisclosed = []
for app_id in bluetooth_apps:
    data = flag_dict.get(app_id, {})
    policy = (data.get("policy_text", "") or "").lower()

    if any(term in policy for term in BLUETOOTH_TERMS):
        bt_disclosed.append(app_id)
    else:
        bluetooth_undisclosed.append(app_id)

print("apps:", len(bluetooth_apps))
print("undisclosed:", len(bluetooth_undisclosed))
print("Disclosed:", len(bt_disclosed))






apps: 40
undisclosed: 40
Disclosed: 0


## Step 1: Permission Label Mapping
Maps semantic category labels to actual Play Store permission strings so comparisons are valid.

In [8]:
# Maps high-level category labels to actual Play Store permission strings
PERMISSION_MAP = {
    "location_tracking": [
        "access precise location",
        "access approximate location"
    ],
    "microphone": [
        "record audio"
    ],
    "camera": [
        "take pictures and videos"
    ],
    "contacts": [
        "read your contacts",
        "find accounts on the device"
    ],
    "storage": [
        "read the contents of your usb storage",
        "modify or delete the contents of your usb storage"
    ],
    "bluetooth": [
        "pair with bluetooth devices",
        "access bluetooth settings"
    ],
    "phone": [
        "read phone status and identity",
        "directly call phone numbers"
    ]
}

# Policy disclosure keywords per category (for cross-referencing)
DISCLOSURE_TERMS = {
    "location_tracking": ["location", "gps", "geolocation", "precise location", "approximate location"],
    "microphone":        ["microphone", "audio", "voice", "record", "speech"],
    "camera":            ["camera", "photo", "picture", "image", "video"],
    "contacts":          ["contacts", "address book", "phone book"],
    "storage":           ["storage", "files", "usb", "external storage"],
    "bluetooth":         ["bluetooth", "ble", "bluetooth low energy", "nearby devices",
                          "device discovery", "scan for devices", "pairing", "beacon"],
    "phone":             ["phone number", "device id", "imei", "call", "telephony"]
}

def cross_reference(permissions_list, policy_text, category):
    """
    Returns True if:
    - app requests a permission in the given category, AND
    - policy discloses it via any relevant disclosure term
    Returns (has_permission, is_disclosed).
    """
    perms_lower = [p.lower() for p in permissions_list]
    mapped_perms = PERMISSION_MAP.get(category, [])
    disclosure_terms = DISCLOSURE_TERMS.get(category, [])

    has_permission = any(mp in perms_lower for mp in mapped_perms)
    is_disclosed = any(term in policy_text for term in disclosure_terms)

    return has_permission, is_disclosed

print("Permission mapping loaded. Categories:", list(PERMISSION_MAP.keys()))

Permission mapping loaded. Categories: ['location_tracking', 'microphone', 'camera', 'contacts', 'storage', 'bluetooth', 'phone']


## Step 2: Results — Disclosure Gap Analysis by Category
For each permission category, report how many apps request it, how many disclose it, and the gap rate.

In [9]:
from collections import defaultdict

# Results storage
category_results = defaultdict(lambda: {"has_perm": 0, "disclosed": 0, "undisclosed": 0, "apps": []})

total_apps = len(flag_dict)
apps_with_any_gap = set()

for app_id, data in flag_dict.items():
    perms = data.get("permissions", [])
    policy = (data.get("policy_text", "") or "").lower()
    title = data.get("title", app_id)

    for category in PERMISSION_MAP:
        has_perm, is_disclosed = cross_reference(perms, policy, category)

        if has_perm:
            category_results[category]["has_perm"] += 1
            if is_disclosed:
                category_results[category]["disclosed"] += 1
            else:
                category_results[category]["undisclosed"] += 1
                category_results[category]["apps"].append(title)
                apps_with_any_gap.add(app_id)

# Print summary
print(f"Total apps analyzed: {total_apps}")
print(f"Apps with at least one disclosure gap: {len(apps_with_any_gap)} ({len(apps_with_any_gap)/total_apps*100:.1f}%)")
print()
print(f"{'Category':<20} {'Has Perm':>10} {'Disclosed':>10} {'Undisclosed':>12} {'Gap Rate':>10}")
print("-" * 65)
for cat, res in sorted(category_results.items()):
    hp = res["has_perm"]
    if hp == 0:
        continue
    gap_rate = res["undisclosed"] / hp * 100
    print(f"{cat:<20} {hp:>10} {res['disclosed']:>10} {res['undisclosed']:>12} {gap_rate:>9.1f}%")


Total apps analyzed: 585
Apps with at least one disclosure gap: 5 (0.9%)

Category               Has Perm  Disclosed  Undisclosed   Gap Rate
-----------------------------------------------------------------
bluetooth                    40         35            5      12.5%


## Step 3: Manual Validation — Precision Estimation
Manually review a sample of flagged violations to estimate precision. Fill in `correct_flags` after reviewing.

In [10]:
import random

# Collect all flagged (app, category) pairs
all_violations = []
for category, res in category_results.items():
    for app_title in res["apps"]:
        all_violations.append((app_title, category))

# Sample 20-30 for manual review
SAMPLE_SIZE = min(25, len(all_violations))
review_sample = random.sample(all_violations, SAMPLE_SIZE)

print(f"Sampled {SAMPLE_SIZE} violations for manual review:")
print(f"{'#':<4} {'App Title':<45} {'Category':<20}")
print("-" * 72)
for i, (title, cat) in enumerate(review_sample, 1):
    print(f"{i:<4} {title[:44]:<45} {cat:<20}")

# --- After manual review, fill in how many were correct ---
# Example: if you reviewed 25 and 19 were true violations:
correct_flags = None  # <-- Replace with your count after review

if correct_flags is not None:
    precision = correct_flags / SAMPLE_SIZE
    print(f"\nManual review: {correct_flags}/{SAMPLE_SIZE} correct")
    print(f"Estimated Precision: {precision*100:.1f}%")
else:
    print("\n[Pending manual review — set correct_flags to compute precision]")


Sampled 5 violations for manual review:
#    App Title                                     Category            
------------------------------------------------------------------------
1    Dr. Panda Town Tales                          bluetooth           
2    Kid-E-Cats Kids Colour Book                   bluetooth           
3    Robocar POLI: Sing Along                      bluetooth           
4    Robocar POLI: Official Video                  bluetooth           
5    Miga Town: My TV Shows                        bluetooth           

[Pending manual review — set correct_flags to compute precision]


## Step 4: Optimized LLM Verification
Instead of checking every sentence, pass only the top 2 highest-scoring sentences from SBERT/NLI to Llama. This is faster and reduces noise.

In [11]:
import ollama
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_top_sentences(policy_text, query, top_k=2):
    """Return the top_k most semantically relevant sentences from policy_text for a given query."""
    sentences = sent_tokenize(policy_text)
    if not sentences:
        return []
    query_emb = sbert_model.encode(query, convert_to_tensor=True)
    sent_embs = sbert_model.encode(sentences, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, sent_embs)[0]
    top_indices = scores.topk(min(top_k, len(sentences))).indices.tolist()
    return [sentences[i] for i in top_indices]

def llama_check(permission, sentence):
    prompt = f"""You are a privacy compliance analyzer.

Permission requested by the app: {permission}
Policy sentence: {sentence}

Does this sentence disclose the permission? Answer YES or NO only."""
    response = ollama.chat(model="llama3", messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"].strip().upper().startswith("YES")

def verify_with_llm(app_id, category, policy_text):
    """
    Uses SBERT to find top 2 relevant sentences, then verifies with Llama.
    Returns True if LLM confirms disclosure.
    """
    query = f"Does this app disclose {category.replace('_', ' ')} data collection?"
    top_sentences = get_top_sentences(policy_text, query, top_k=2)
    permission_label = category.replace("_", " ")

    for sentence in top_sentences:
        if llama_check(permission_label, sentence):
            return True
    return False

print("Optimized LLM verification functions loaded.")
print("Usage: verify_with_llm(app_id, 'location_tracking', policy_text)")


c:\Users\rohit\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7763.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Optimized LLM verification functions loaded.
Usage: verify_with_llm(app_id, 'location_tracking', policy_text)


## Step 5: Baseline Comparison — Keyword Matching vs. NLP Pipeline
Compare simple keyword matching against the SBERT+NLI+LLM pipeline to show improvement.

In [12]:
# Keyword baseline: just check if the category name appears in the policy
def keyword_baseline(permissions_list, policy_text, category):
    """Naive baseline: checks if a single keyword for the category appears in the policy."""
    keyword_map = {
        "location_tracking": "location",
        "microphone":        "microphone",
        "camera":            "camera",
        "contacts":          "contacts",
        "storage":           "storage",
        "bluetooth":         "bluetooth",
        "phone":             "phone"
    }
    perms_lower = [p.lower() for p in permissions_list]
    mapped_perms = PERMISSION_MAP.get(category, [])
    has_perm = any(mp in perms_lower for mp in mapped_perms)
    keyword = keyword_map.get(category, category)
    is_disclosed = keyword in policy_text
    return has_perm, is_disclosed

# Run baseline over full dataset
baseline_gaps = defaultdict(int)
nlp_gaps = defaultdict(int)

for app_id, data in flag_dict.items():
    perms = data.get("permissions", [])
    policy = (data.get("policy_text", "") or "").lower()

    for category in PERMISSION_MAP:
        has_perm_b, disclosed_b = keyword_baseline(perms, policy, category)
        has_perm_n, disclosed_n = cross_reference(perms, policy, category)

        if has_perm_b and not disclosed_b:
            baseline_gaps[category] += 1
        if has_perm_n and not disclosed_n:
            nlp_gaps[category] += 1

print(f"{'Category':<20} {'Keyword Gaps':>14} {'NLP Gaps':>10} {'Difference':>12}")
print("-" * 60)
total_baseline, total_nlp = 0, 0
for cat in PERMISSION_MAP:
    b = baseline_gaps[cat]
    n = nlp_gaps[cat]
    diff = b - n  # positive = NLP catches fewer false positives (more precise)
    total_baseline += b
    total_nlp += n
    print(f"{cat:<20} {b:>14} {n:>10} {diff:>+12}")

print("-" * 60)
print(f"{'TOTAL':<20} {total_baseline:>14} {total_nlp:>10} {total_baseline - total_nlp:>+12}")
print()
if total_baseline > 0:
    improvement = (total_baseline - total_nlp) / total_baseline * 100
    print(f"NLP pipeline reduces noisy gaps by {improvement:.1f}% compared to keyword baseline.")

Category               Keyword Gaps   NLP Gaps   Difference
------------------------------------------------------------
location_tracking                 0          0           +0
microphone                        0          0           +0
camera                            0          0           +0
contacts                          0          0           +0
storage                           0          0           +0
bluetooth                        40          5          +35
phone                             0          0           +0
------------------------------------------------------------
TOTAL                            40          5          +35

NLP pipeline reduces noisy gaps by 87.5% compared to keyword baseline.


## Step 6: Model Comparison — SBERT Variants
Compare `all-MiniLM-L6-v2` vs `all-mpnet-base-v2` to see which better detects disclosure. Also compare with/without LLM verification.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from nltk.tokenize import sent_tokenize
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
MODELS_TO_COMPARE = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2"
]

SIMILARITY_THRESHOLD = 0.45  # tune as needed

def sbert_discloses(policy_text, category, model, threshold=SIMILARITY_THRESHOLD):
    """Returns True if any sentence in the policy is semantically similar to the category query."""
    query = f"This app collects {category.replace('_', ' ')} data from users."
    sentences = sent_tokenize(policy_text)
    if not sentences:
        return False
    query_emb = model.encode(query, convert_to_tensor=True)
    sent_embs = model.encode(sentences, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, sent_embs)[0]
    return float(scores.max()) >= threshold

model_gap_counts = {name: defaultdict(int) for name in MODELS_TO_COMPARE}
loaded_models = {name: SentenceTransformer(name) for name in MODELS_TO_COMPARE}

for app_id, data in flag_dict.items():
    perms = data.get("permissions", [])
    policy = (data.get("policy_text", "") or "").lower()

    for category in PERMISSION_MAP:
        mapped_perms = PERMISSION_MAP.get(category, [])
        perms_lower = [p.lower() for p in perms]
        has_perm = any(mp in perms_lower for mp in mapped_perms)
        if not has_perm:
            continue

        for model_name, model in loaded_models.items():
            disclosed = sbert_discloses(policy, category, model)
            if not disclosed:
                model_gap_counts[model_name][category] += 1

print(f"{'Category':<20}", end="")
for name in MODELS_TO_COMPARE:
    short = name.split("/")[-1][:18]
    print(f"  {short:>18}", end="")
print()
print("-" * (20 + 20 * len(MODELS_TO_COMPARE)))

for cat in PERMISSION_MAP:
    print(f"{cat:<20}", end="")
    for name in MODELS_TO_COMPARE:
        print(f"  {model_gap_counts[name][cat]:>18}", end="")
    print()

print()
print("Lower gap count = model detects more disclosures (potentially more sensitive).")
print("Higher gap count = model flags more potential violations.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\rohit\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7545.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7132.32it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - 'C:\\Users\\rohit/nltk_data'
    - 'c:\\Users\\rohit\\miniconda3\\nltk_data'
    - 'c:\\Users\\rohit\\miniconda3\\share\\nltk_data'
    - 'c:\\Users\\rohit\\miniconda3\\lib\\nltk_data'
    - 'C:\\Users\\rohit\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************
